In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns


## 1. EDA

In [2]:
DATA_SOURCE = '../../../data/legal_document/metadata_law_ALL.json'
OUTPUT_FILTERED = '../../../data/legal_document/metadata_law_ALL_FILTERED.json'

df = pd.read_json(DATA_SOURCE)
df.describe()

,Tên văn bản,Số hiệu,Ngành,Lĩnh vực,Cơ quan ban hành,Chức danh,Người ký,Loại văn bản,Ngày ban hành,Ngày có hiệu lực,Ngày hết hiệu lực,Tình trạng hiệu lực,url,nội dung,Ngày ký xác thực
count,161,161,161,161,160,160,160,161,159,161,161,159,161,161,2
unique,161,160,23,30,39,16,79,10,146,140,2,3,161,161,2
top,"Luật Sửa đổi, bổ sung một số điều của Luật Doa...",59/2020/QH14,--,Chưa phân loại,Chính phủ,Thủ tướng,Nguyễn Tấn Dũng,Quyết định,10/05/2013,01/07/2013,--,Còn hiệu lực,https://vbpl.vn/van-ban/chi-tiet/luat-sua-doi-...,QUỐC HỘI\nLuật số: 76/2025/QH15\nCỘNG HÒA XÃ H...,04/06/2026
freq,1,2,63,106,42,37,26,47,4,5,160,129,1,1,1


In [3]:
temp_df = df.copy()
temp_df['contains_keyword'] = temp_df['Tên văn bản'].str.contains('luật|sửa đổi', case=False, na=False)
print("Type of documents:")
print(temp_df['Loại văn bản'].value_counts())

Type of documents:
Loại văn bản
Quyết định            47
Nghị định             42
Thông tư              23
Chỉ thị               14
Luật                  12
Lệnh                  10
Nghị quyết             6
Thông tư liên tịch     4
Văn bản hợp nhất       2
Bộ luật                1
Name: count, dtype: int64


## 2. General processing

### 2.1. Applying processing techniques

In [4]:
# Display unique expiration statuses and document types for inspection
print(f"Types of expiration ({len(df['Tình trạng hiệu lực'].unique())}): {df['Tình trạng hiệu lực'].unique()}")
print(f"Types of documents ({len(df['Loại văn bản'].unique())}): {df['Loại văn bản'].unique()}")

# Apply filters step by step
filtered_df = df[
    df['Tình trạng hiệu lực'].isin([
        'Còn hiệu lực', 
        'Hết hiệu lực một phần', 
        # 'Chưa có hiệu lực'
    ])
]

filtered_df = filtered_df[
    filtered_df['Loại văn bản'].isin([
        'Luật', 
        # 'Nghị định'
    ])
]

filtered_df = filtered_df[
    filtered_df['Tên văn bản'].str.contains('luật|sửa đổi|nghị định', case=False, na=False)
]

Types of expiration (4): ['Còn hiệu lực' 'Hết hiệu lực một phần' None 'Chưa có hiệu lực']
Types of documents (10): ['Luật' 'Nghị định' 'Thông tư' 'Quyết định' 'Thông tư liên tịch' 'Lệnh'
 'Nghị quyết' 'Chỉ thị' 'Văn bản hợp nhất' 'Bộ luật']


In [5]:
print(f"After filtering: {len(filtered_df)}")
print(f"Types of expiration ({len(filtered_df['Tình trạng hiệu lực'].unique())}): {filtered_df['Tình trạng hiệu lực'].unique()}")
print(f"Types of documents ({len(filtered_df['Loại văn bản'].unique())}): {filtered_df['Loại văn bản'].unique()}")

After filtering: 12
Types of expiration (2): ['Còn hiệu lực' 'Hết hiệu lực một phần']
Types of documents (1): ['Luật']


### 2.2. Preprocessed table

In [6]:
print(f"Number of documents after filtering: {len(filtered_df)}")
(filtered_df[['Tên văn bản', 'Số hiệu', 'Loại văn bản', 'Tình trạng hiệu lực', 'nội dung']].head())

Number of documents after filtering: 12


,Tên văn bản,Số hiệu,Loại văn bản,Tình trạng hiệu lực,nội dung
0,"Luật Sửa đổi, bổ sung một số điều của Luật Doa...",76/2025/QH15,Luật,Còn hiệu lực,QUỐC HỘI\nLuật số: 76/2025/QH15\nCỘNG HÒA XÃ H...
1,Luật Thuế thu nhập doanh nghiệp số 67/2025/QH15,67/2025/QH15,Luật,Hết hiệu lực một phần,QUỐC HỘI\nLuật số: 67/2025/QH15\nCỘNG HÒA XÃ H...
2,Luật Quản lý và đầu tư vốn nhà nước tại doanh ...,68/2025/QH15,Luật,Còn hiệu lực,QUỐC HỘI\n-------\nCỘNG HÒA XÃ HỘI CHỦ NGHĨA V...
3,"Luật sửa đổi, bổ sung một số điều của Luật Đầu...",03/2022/QH15,Luật,Hết hiệu lực một phần,QUỐC HỘI\n\nCỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM...
4,Luật Doanh nghiệp số 59/2020/QH14,59/2020/QH14,Luật,Còn hiệu lực,QUỐC HỘI\n------- CỘNG HÒA XÃ HỘI CHỦ NGHĨA VI...


In [7]:
filtered_df.to_json(
    OUTPUT_FILTERED,
    orient='records',      # each row becomes a JSON object
    force_ascii=False,     # keep Vietnamese characters
    indent=4               # pretty print with 4 spaces
)

print(f"Exported {len(filtered_df)} records to {OUTPUT_FILTERED}")

Exported 12 records to ../../../data/legal_document/metadata_law_ALL_FILTERED.json


In [8]:
text = filtered_df.iloc[0]['nội dung']
print(text)

QUỐC HỘI
Luật số: 76/2025/QH15
CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM
Độc lập - Tự do - Hạnh phúc
-----------------------------
LUẬT
SỬA ĐỔI, BỔ SUNG MỘT SỐ ĐIỀU CỦA LUẬT DOANH NGHIỆP
Căn cứ Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam đã được sửa đổi, bổ sung một số điều theo Nghị quyết số 203/2025/QH15;
Quốc hội ban hành Luật sửa đổi, bổ sung một số điều của Luật Doanh nghiệp số 59/2020/QH14 đã được sửa đổi, bổ sung một số điều theo Luật số 03/2022/QH15.
Điều 1. Sửa đổi, bổ sung Luật Doanh nghiệp
1. Sửa đổi, bổ sung một số khoản của Điều 4 như sau:
a) Sửa đổi, bổ sung khoản 5 như sau:
"5. Cổ tức là khoản lợi nhuận sau thuế được trả cho mỗi cổ phần bằng tiền hoặc bằng tài sản khác.";
b) Sửa đổi, bổ sung khoản 14 như sau:
"14. Giá thị trường của phần vốn góp hoặc cổ phần là:
a) Giá giao dịch bình quân trong vòng 30 ngày liền kề trước ngày xác định giá hoặc giá thỏa thuận giữa người bán và người mua hoặc giá do một tổ chức thẩm định giá xác định đối với cổ phiếu niêm yết, đăng ký giao 